# PneumoScan — EfficientNet-B0 Transfer Learning

This notebook applies **EfficientNet-B0** as a feature extractor for pneumonia classification on chest X-rays.  
EfficientNet uses a **compound scaling** strategy that uniformly scales network width, depth, and resolution with a set of fixed coefficients, achieving superior accuracy-per-FLOP compared to manually designed architectures.  
The B0 variant is the baseline of the family—compact enough for rapid iteration yet powerful enough to serve as a strong transfer-learning backbone.  

**Classes:** `NORMAL`, `BACTERIA`, `VIRUS` &nbsp;|&nbsp; **Input size:** 224 × 224 px

In [ ]:
import sys, os

# Make the src/ package importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src import config, data_loader, models, train, evaluate

# Colab / environment helpers
config.setup_colab()
config.ensure_dirs()

print("Environment ready.")
print(f"Classes : {config.CLASS_NAMES}")
print(f"Image size : {config.IMAGE_SIZE}")

## 1 — Data Loading

We load the training and validation splits produced by the project’s `data_loader` module and compute per-class weights to handle the inherent class imbalance in chest X-ray datasets.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load train / validation datasets
train_ds, val_ds = data_loader.load_train_val_split()

# Compute class weights from training labels
train_labels = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
if train_labels.ndim > 1:          # one-hot → integer labels
    train_labels = np.argmax(train_labels, axis=1)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(config.CLASS_NAMES)),
    y=train_labels,
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights:")
for idx, name in enumerate(config.CLASS_NAMES):
    print(f"  {name:>10s}: {class_weights[idx]:.4f}")

## 2 — Model Architecture

### Why EfficientNet-B0?

EfficientNet introduced the idea of **compound scaling**: instead of arbitrarily increasing only depth, width, or resolution, a single compound coefficient $\phi$ governs all three dimensions simultaneously:

| Dimension | Scaling Rule |
|-----------|-------------|
| Depth     | $d = \alpha^{\phi}$ |
| Width     | $w = \beta^{\phi}$  |
| Resolution| $r = \gamma^{\phi}$ |

where $\alpha$, $\beta$, $\gamma$ are determined by a small grid search on the baseline network.  
B0 ($\phi = 0$) is the baseline—only **5.3 M** parameters—yet it matches or exceeds the accuracy of much larger models.  
For medical imaging tasks with limited data, B0 provides an excellent accuracy–compute trade-off and reduces the risk of overfitting compared to heavier variants.

In [ ]:
# Build EfficientNet-B0 with frozen ImageNet backbone
model = models.build_efficientnet_b0(freeze=True)
model.summary()

## 3 — Training

We train the classification head on top of the frozen EfficientNet-B0 features. Callbacks such as early stopping, learning-rate reduction, and model checkpointing are handled inside `train.train_model()`.

In [ ]:
history = train.train_model(
    model_name="efficientnet_b0",
    train_ds=train_ds,
    val_ds=val_ds,
    class_weights=class_weights,
)

print("Training complete.")

## 4 — Evaluation

We load the held-out test set and run `evaluate.evaluate_model()` which computes the classification report, confusion matrix, and other diagnostic plots.

In [ ]:
# Load test dataset
test_ds = data_loader.load_test_data()

# Full evaluation (metrics + plots)
results = evaluate.evaluate_model(
    model=model,
    test_ds=test_ds,
    model_name="efficientnet_b0",
)

print("Test evaluation finished.")

## Summary

EfficientNet-B0 leverages compound scaling to deliver strong feature representations in a compact model.  
Key take-aways from this experiment:

* **Parameter efficiency** — With only ~5.3 M parameters (and far fewer trainable when the backbone is frozen), EfficientNet-B0 trains quickly even on free-tier Colab GPUs.
* **Transfer-learning friendly** — ImageNet pre-trained weights generalise well to grayscale chest X-rays after minimal fine-tuning of the classification head.
* **Balanced accuracy** — Class weights help the model cope with the skewed distribution typical of medical datasets (more BACTERIA samples than VIRUS).

Next steps include comparing these results against DenseNet-121 and MobileNetV2 transfer-learning baselines.